In [5]:
import pandas as pd
import sqlite3
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2009-2010')
df2 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2010-2011')
df = pd.concat([df, df2], ignore_index=True)

print(df.shape)
print(df.head())
print(df.dtypes)

(1067371, 8)
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom  
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           floa

In [9]:
# Clean data
df.columns = ['invoice', 'stock_code', 'description', 'quantity', 
               'invoice_date', 'price', 'customer_id', 'country']

# Remove cancellations, nulls, and bad rows
df = df[df['quantity'] > 0]
df = df[df['price'] > 0]
df = df.dropna(subset=['customer_id'])
df['customer_id'] = df['customer_id'].astype(int).astype(str)
df['revenue'] = df['quantity'] * df['price']

print("Clean data shape:", df.shape)
print("Date range:", df['invoice_date'].min(), "to", df['invoice_date'].max())
print("Unique customers:", df['customer_id'].nunique())

# Load into SQLite
conn = sqlite3.connect('retail.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)
print("Loaded into SQLite successfully!")

Clean data shape: (805549, 9)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5878
Loaded into SQLite successfully!


In [10]:
# SQL Analysis
import pandas as pd

# 1. Revenue by month
monthly_revenue = pd.read_sql_query("""
    SELECT 
        strftime('%Y-%m', invoice_date) AS month,
        ROUND(SUM(revenue), 2) AS total_revenue,
        COUNT(DISTINCT invoice) AS num_orders,
        COUNT(DISTINCT customer_id) AS active_customers
    FROM transactions
    GROUP BY month
    ORDER BY month
""", conn)

print("Monthly Revenue:")
print(monthly_revenue)

Monthly Revenue:
      month  total_revenue  num_orders  active_customers
0   2009-12      686654.16        1512               955
1   2010-01      557319.06        1011               720
2   2010-02      506371.07        1104               772
3   2010-03      699608.99        1524              1057
4   2010-04      594609.19        1329               942
5   2010-05      599985.79        1377               966
6   2010-06      639066.58        1497              1041
7   2010-07      591636.74        1381               928
8   2010-08      604242.65        1293               911
9   2010-09      831615.00        1689              1145
10  2010-10     1036680.00        2133              1497
11  2010-11     1172336.04        2587              1607
12  2010-12      884591.89        1400               885
13  2011-01      569445.04         987               741
14  2011-02      447137.35         997               758
15  2011-03      595500.76        1321               974
16  2011-04   

In [11]:
# 2. Top 10 products by revenue
top_products = pd.read_sql_query("""
    SELECT 
        description,
        SUM(quantity) AS total_units,
        ROUND(SUM(revenue), 2) AS total_revenue
    FROM transactions
    GROUP BY description
    ORDER BY total_revenue DESC
    LIMIT 10
""", conn)

print("Top 10 Products:")
print(top_products)

Top 10 Products:
                          description  total_units  total_revenue
0            REGENCY CAKESTAND 3 TIER        24899      286486.30
1  WHITE HANGING HEART T-LIGHT HOLDER        93640      252072.46
2         PAPER CRAFT , LITTLE BIRDIE        80995      168469.60
3                              Manual         9803      152340.57
4             JUMBO BAG RED RETROSPOT        75759      136980.08
5       ASSORTED COLOUR BIRD ORNAMENT        79913      127074.17
6                             POSTAGE         5333      126563.04
7                       PARTY BUNTING        23607      103880.23
8      MEDIUM CERAMIC TOP STORAGE JAR        77916       81416.73
9     PAPER CHAIN KIT 50'S CHRISTMAS         29477       79594.33


In [12]:
# 3. Repeat vs one-time customers
customer_segments = pd.read_sql_query("""
    WITH customer_orders AS (
        SELECT customer_id, COUNT(DISTINCT invoice) AS order_count
        FROM transactions
        GROUP BY customer_id
    )
    SELECT 
        CASE WHEN order_count = 1 THEN 'One-time' ELSE 'Repeat' END AS customer_type,
        COUNT(*) AS customers,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
    FROM customer_orders
    GROUP BY customer_type
""", conn)

print("Customer Segments:")
print(customer_segments)

Customer Segments:
  customer_type  customers    pct
0      One-time       1623  27.61
1        Repeat       4255  72.39


In [13]:
# 4. Cancellation rate by country
cancellations = pd.read_sql_query("""
    SELECT
        country,
        COUNT(DISTINCT invoice) AS total_orders,
        SUM(CASE WHEN invoice LIKE 'C%' THEN 1 ELSE 0 END) AS cancellations,
        ROUND(100.0 * SUM(CASE WHEN invoice LIKE 'C%' THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancel_rate
    FROM transactions
    GROUP BY country
    HAVING COUNT(*) > 100
    ORDER BY cancel_rate DESC
    LIMIT 10
""", conn)

print("Cancellation Rates:")
print(cancellations)

Cancellation Rates:
                country  total_orders  cancellations  cancel_rate
0           Unspecified            16              0          0.0
1        United Kingdom         33541              0          0.0
2  United Arab Emirates            11              0          0.0
3                   USA            20              0          0.0
4           Switzerland            90              0          0.0
5                Sweden           104              0          0.0
6                 Spain           154              0          0.0
7             Singapore            11              0          0.0
8                   RSA             2              0          0.0
9              Portugal            93              0          0.0


In [14]:
# RFM + K-Means Clustering
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Build RFM table
snapshot_date = df['invoice_date'].max()

rfm = df.groupby('customer_id').agg(
    recency=('invoice_date', lambda x: (snapshot_date - x.max()).days),
    frequency=('invoice', 'nunique'),
    monetary=('revenue', 'sum')
).reset_index()

print("RFM Table:")
print(rfm.describe())

# Scale and cluster
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency', 'frequency', 'monetary']])

kmeans = KMeans(n_clusters=4, random_state=42)
rfm['cluster'] = kmeans.fit_predict(rfm_scaled)

# Label clusters
cluster_summary = rfm.groupby('cluster').agg(
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    customer_count=('customer_id', 'count')
).round(2)

print("\nCluster Summary:")
print(cluster_summary)

RFM Table:
           recency    frequency       monetary
count  5878.000000  5878.000000    5878.000000
mean    200.331916     6.289384    3018.616737
std     209.338707    13.009406   14737.731040
min       0.000000     1.000000       2.950000
25%      25.000000     1.000000     348.762500
50%      95.000000     3.000000     898.915000
75%     379.000000     7.000000    2307.090000
max     738.000000   398.000000  608821.650000

Cluster Summary:
         avg_recency  avg_frequency  avg_monetary  customer_count
cluster                                                          
0             462.03           2.21        765.24            1998
1              66.01           7.31       3009.40            3841
2              24.94         103.71      83086.08              35
3               2.50         212.50     436835.79               4


In [15]:
import plotly.express as px
import plotly.graph_objects as go

# Label clusters
cluster_labels = {0: 'Lost', 1: 'Loyal', 2: 'VIP', 3: 'Champions'}
rfm['segment'] = rfm['cluster'].map(cluster_labels)

# 1. Customer segments bubble chart
fig1 = px.scatter(rfm, x='recency', y='frequency', size='monetary',
                  color='segment', hover_data=['customer_id'],
                  title='Customer Segments: RFM Analysis',
                  labels={'recency': 'Days Since Last Purchase', 'frequency': 'Number of Orders'},
                  size_max=40)
fig1.write_html('customer_segments.html')
fig1.show()

# 2. Monthly revenue trend
fig2 = px.line(monthly_revenue, x='month', y='total_revenue',
               title='Monthly Revenue Trend (2009-2011)',
               labels={'month': 'Month', 'total_revenue': 'Revenue (£)'},
               markers=True)
fig2.write_html('monthly_revenue.html')
fig2.show()

# 3. Top 10 products
fig3 = px.bar(top_products, x='total_revenue', y='description',
              orientation='h',
              title='Top 10 Products by Revenue',
              labels={'total_revenue': 'Revenue (£)', 'description': 'Product'})
fig3.update_layout(yaxis={'categoryorder': 'total ascending'})
fig3.write_html('top_products.html')
fig3.show()

print("All charts saved!")

All charts saved!


In [18]:
import sqlite3
import pandas as pd
conn = sqlite3.connect('retail.db')
# Export data for Tableau
monthly_revenue = pd.read_sql_query("""
    SELECT 
        strftime('%Y-%m', invoice_date) AS month,
        ROUND(SUM(revenue), 2) AS total_revenue,
        COUNT(DISTINCT invoice) AS num_orders,
        COUNT(DISTINCT customer_id) AS active_customers
    FROM transactions
    GROUP BY month
    ORDER BY month
""", conn)

top_products = pd.read_sql_query("""
    SELECT description, SUM(quantity) AS total_units,
        ROUND(SUM(revenue), 2) AS total_revenue
    FROM transactions
    GROUP BY description
    ORDER BY total_revenue DESC
    LIMIT 10
""", conn)

country_summary = pd.read_sql_query("""
    SELECT country,
        COUNT(DISTINCT customer_id) AS customers,
        COUNT(DISTINCT invoice) AS orders,
        ROUND(SUM(revenue), 2) AS total_revenue
    FROM transactions
    GROUP BY country
    ORDER BY total_revenue DESC
""", conn)

monthly_revenue.to_csv('tableau_monthly_revenue.csv', index=False)
top_products.to_csv('tableau_top_products.csv', index=False)
country_summary.to_csv('tableau_country_summary.csv', index=False)

print("Exported 4 CSV files for Tableau!")

Exported 4 CSV files for Tableau!
